# 🦷 Smile Therapy — AI Maxillofacial Radiology Backend (Google Colab)
This notebook runs the **full original multimodal research model** (DenseNet-121 + 31 Morphometrics + GBDT + Grad-CAM) on Google Colab with free GPU acceleration, and generates a public URL for your Vercel frontend.

In [ ]:
# Step 1: Check GPU
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✓ GPU Device:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Tip: Change runtime to T4 GPU in Runtime -> Change runtime type for ultra-fast GPU inference!")

In [ ]:
# Step 2: Install FastAPI & Cloudflare Tunnel
# Note: Google Colab already has torch, torchvision, scikit-learn, joblib preinstalled.
!pip install -q fastapi uvicorn python-multipart opencv-python-headless pyngrok
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✓ FastAPI, Uvicorn, and Cloudflared installed successfully!")

In [ ]:
# Step 3: Set Directory & Verify Files
import os
if os.path.exists("/content/web_dental/app/backend"):
    %cd /content/web_dental/app/backend
elif os.path.exists("/content/web_dental"):
    %cd /content/web_dental
elif os.path.exists("/content/app/backend"):
    %cd /content/app/backend
else:
    %cd /content

print("Current directory:", os.getcwd())
has_main = os.path.exists("main.py")
has_ckpt = os.path.exists("final_multimodal_checkpoint.joblib")
print("main.py found:", has_main)
print("final_multimodal_checkpoint.joblib found:", has_ckpt)
if not (has_main and has_ckpt):
    print("⚠️ WARNING: Please upload main.py and final_multimodal_checkpoint.joblib into this folder!")

In [ ]:
# Step 4: Run Backend with Original Model and Launch Cloudflare Tunnel
import subprocess, time, re, urllib.request, sys

!pkill -9 -f uvicorn
!pkill -9 -f cloudflared
time.sleep(1)

print("[*] Starting FastAPI Backend with original DenseNet-121 on GPU...")
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

ready = False
for i in range(35):
    # Check if process exited unexpectedly
    if backend_proc.poll() is not None:
        print("❌ Backend failed to start! Error log:")
        print(backend_proc.stdout.read())
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=1) as resp:
            if resp.status == 200:
                ready = True
                print("✓ Backend & Model initialized successfully!")
                break
    except Exception:
        time.sleep(1)

if not ready and backend_proc.poll() is None:
    print("[!] Backend is taking longer to initialize. Checking process output:")
    # Non-blocking read
    import fcntl, os
    try:
        fl = fcntl.fcntl(backend_proc.stdout.fileno(), fcntl.F_GETFL)
        fcntl.fcntl(backend_proc.stdout.fileno(), fcntl.F_SETFL, fl | os.O_NONBLOCK)
        out = backend_proc.stdout.read()
        if out:
            print(out)
    except Exception:
        pass

print("[*] Opening Cloudflare Public Tunnel...")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stdout.readline()
    if not line:
        continue
    match = pattern.search(line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*65)
    print("  🎉 SMILE THERAPY AI BACKEND IS LIVE ON GOOGLE COLAB! 🎉")
    print("="*65)
    print(f"\n  👉 PUBLIC API URL:   {tunnel_url}")
    print(f"  👉 HEALTH CHECK:     {tunnel_url}/health\n")
    print("  📋 Copy the URL above and paste it in your Vercel frontend Server Settings!")
    print("="*65 + "\n")
else:
    print("[!] Cloudflare URL not detected. Check console logs:")

# Keep cell alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping services...")
    backend_proc.terminate()
    cf_proc.terminate()


In [ ]:
# Step 5 (Optional Alternative): ngrok Tunnel (If Cloudflare is blocked in your network)
# Sign up free at https://dashboard.ngrok.com to get your Auth Token
# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_TOKEN_HERE")
# ngrok_tunnel = ngrok.connect(8000)
# print("👉 NGROK PUBLIC URL:", ngrok_tunnel.public_url)
